
# NER: Silver-only, Silver → Gold, and Gold-only (with Early Stopping)

This notebook trains and compares three token-classification (NER) models:

1. **Silver-only**: train on silver annotations only.
2. **Silver → Gold**: initialize from the silver-only model and fine-tune on the gold split.
3. **Gold-only**: train from scratch (same base model) on the gold split only.

All models use **early stopping** and evaluate on the **gold test set** for an apples-to-apples comparison.
If separate silver test data is present, the silver-only section will also report metrics on it.

**Data expectations**

- **Gold (new files, required)**:  
  - `splits/train_val.json` — list of items with `{"tokens": [...], "ner_tags": [...]}`  
  - `splits/test.json`

- **Silver (optional; "keep using these")**:  
  - `silver/train_val.json`  
  - `silver/test.json`

If the silver files are not present, those sections will be skipped with a friendly warning.


In [ ]:

import os, json, random, numpy as np
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional

from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---- Paths ----
ROOT = Path(".")
LLM_DIR = ROOT / "llm" / "bio"

# Gold (use the new split files you generated)
GOLD_TRAINVAL_PATH = ROOT / "splits" / "train_val.json"
GOLD_TEST_PATH     = ROOT / "splits" / "test.json"

# Silver: multiple JSONs live directly under LLM_DIR (no silver test; we'll evaluate on GOLD test)
SILVER_FILES_PATTERN = "*.json" 

# ---- Model & Training Hyperparams ----
BASE_MODEL   = "GroNLP/bert-base-dutch-cased"
MAX_LEN      = 512
BATCH_SIZE   = 8
LEARNING_RATE= 5e-5
EPOCHS       = 10
PATIENCE     = 3
VAL_RATIO    = 0.1
LOG_STEPS    = 50

OUTDIR_SILVER_ONLY    = Path("models/silver_only_xlmr")
OUTDIR_SILVER_TO_GOLD = Path("models/silver_to_gold_xlmr")
OUTDIR_GOLD_ONLY      = Path("models/gold_only_xlmr")

for p in [OUTDIR_SILVER_ONLY, OUTDIR_SILVER_TO_GOLD, OUTDIR_GOLD_ONLY]:
    p.mkdir(parents=True, exist_ok=True)

def load_json_list(path: Path) -> list:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f"{path} must contain a list of examples.")
    return data

def build_label_space(*datasets: List[Dict[str, Any]]) -> Tuple[int, Dict[int, str], Dict[str, int]]:
    all_labels = set()
    for ds in datasets:
        for ex in ds:
            all_labels.update(ex["ner_tags"])
    num_labels = max(all_labels) + 1
    id2label = {i: f"L{i}" for i in range(num_labels)}
    label2id = {v: k for k, v in id2label.items()}
    return num_labels, id2label, label2id

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

def tokenize_and_align(example: Dict[str, Any]) -> Dict[str, Any]:
    tokens = example["tokens"]
    labels = example["ner_tags"]
    enc = tokenizer(tokens, is_split_into_words=True, truncation=True, max_length=MAX_LEN)
    word_ids = enc.word_ids()

    aligned = []
    prev_word_id = None
    for wid in word_ids:
        if wid is None:
            aligned.append(-100)
        else:
            if wid != prev_word_id:
                aligned.append(labels[wid])
            else:
                aligned.append(-100)
        prev_word_id = wid
    enc["labels"] = aligned
    return enc

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    labels = p.label_ids
    true_predictions = []
    true_labels = []
    for pred, lab in zip(preds, labels):
        cur_preds, cur_labels = [], []
        for p_id, l_id in zip(pred, lab):
            if l_id != -100:
                cur_preds.append(id2label[p_id])
                cur_labels.append(id2label[l_id])
        true_predictions.append(cur_preds)
        true_labels.append(cur_labels)
    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

def make_trainer(output_dir: Path, num_labels: int, id2label: Dict[int,str], label2id: Dict[str,int],
                 train_ds: Dataset, val_ds: Dataset, model_name_or_path: str):
    model = AutoModelForTokenClassification.from_pretrained(
        model_name_or_path,
        num_labels=num_labels, id2label=id2label, label2id=label2id
    )
    args = TrainingArguments(
        output_dir=str(output_dir),
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        logging_steps=LOG_STEPS,
        eval_steps=LOG_STEPS,
        save_steps=LOG_STEPS,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        seed=SEED,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)]
    )
    return trainer

def make_trainer_silver(output_dir: Path, num_labels: int, id2label: Dict[int,str], label2id: Dict[str,int],
                        train_ds: Dataset, val_ds: Dataset, model_name_or_path: str):
    model = AutoModelForTokenClassification.from_pretrained(
        model_name_or_path,
        num_labels=num_labels, id2label=id2label, label2id=label2id
    )
    args = TrainingArguments(
        output_dir=str(output_dir),
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        num_train_epochs=2,   # <---- only 2 epochs for silver
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        logging_steps=LOG_STEPS,
        eval_steps=LOG_STEPS,
        save_steps=LOG_STEPS,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        seed=SEED,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)]
    )
    return trainer

def evaluate_report(trainer: "Trainer", test_ds: Dataset, id2label: Dict[int,str]) -> dict:
    metrics = trainer.evaluate(test_ds)
    preds = np.argmax(trainer.predict(test_ds).predictions, axis=-1)
    labels = test_ds["labels"]
    true_predictions, true_labels = [], []
    for pred, lab in zip(preds, labels):
        cur_preds, cur_labels = [], []
        for p_id, l_id in zip(pred, lab):
            if l_id != -100:
                cur_preds.append(id2label[p_id])
                cur_labels.append(id2label[l_id])
        true_predictions.append(cur_preds)
        true_labels.append(cur_labels)

    print("\nSeqeval classification report:")
    print(classification_report(true_labels, true_predictions, digits=4))
    return metrics


In [8]:

# ---- Load GOLD (required) ----
if not GOLD_TRAINVAL_PATH.exists() or not GOLD_TEST_PATH.exists():
    raise FileNotFoundError("Gold files not found. Expected splits/train_val.json and splits/test.json")

gold_trainval = load_json_list(GOLD_TRAINVAL_PATH)
gold_test     = load_json_list(GOLD_TEST_PATH)

# split gold train/val
gold_train_list, gold_val_list = train_test_split(gold_trainval, test_size=VAL_RATIO, random_state=SEED, shuffle=True)

# build labels from gold sets (and optionally silver later)
num_labels, id2label, label2id = build_label_space(gold_train_list, gold_val_list, gold_test)

# tokenize
gold_train_ds = Dataset.from_list(gold_train_list).map(tokenize_and_align, batched=False)
gold_val_ds   = Dataset.from_list(gold_val_list).map(tokenize_and_align, batched=False)
gold_test_ds  = Dataset.from_list(gold_test).map(tokenize_and_align, batched=False)

len(gold_train_ds), len(gold_val_ds), len(gold_test_ds)


Map:   0%|          | 0/794 [00:00<?, ? examples/s]

Map: 100%|██████████| 184/184 [00:00<00:00, 3734.48 examples/s]


(794, 89, 184)

In [9]:
# ---- Load SILVER (train/val come from concatenating multiple JSON files in LLM_DIR) ----
silver_files = sorted(LLM_DIR.glob(SILVER_FILES_PATTERN))
have_silver = len(silver_files) > 0

if have_silver:
    silver_all = []
    for f in silver_files:
        with open(f, "r", encoding="utf-8") as infile:
            content = json.load(infile)
            if isinstance(content, list):
                silver_all.extend(content)
            else:
                silver_all.append(content)

    if not silver_all:
        print("Warning: silver files found but contain no items; skipping silver-only and silver→gold.")
        have_silver = False
    else:
        print(len(silver_all))
else:
    print(f"No silver files found in {LLM_DIR} matching pattern '{SILVER_FILES_PATTERN}'. Skipping silver-only and silver→gold.")

7228


In [10]:
# ---- Label space from GOLD (and from SILVER if present) ----
if have_silver:
    num_labels, id2label, label2id = build_label_space(gold_train_list, gold_val_list, gold_test, silver_all)
else:
    num_labels, id2label, label2id = build_label_space(gold_train_list, gold_val_list, gold_test)

# ---- Tokenize GOLD ----
gold_train_ds = Dataset.from_list(gold_train_list).map(tokenize_and_align, batched=False)
gold_val_ds   = Dataset.from_list(gold_val_list).map(tokenize_and_align, batched=False)
gold_test_ds  = Dataset.from_list(gold_test).map(tokenize_and_align, batched=False)

# ---- Tokenize SILVER (if present) ----
if have_silver:
    silver_train_list, silver_val_list = train_test_split(silver_all, test_size=VAL_RATIO, random_state=SEED, shuffle=True)
    silver_train_ds = Dataset.from_list(silver_train_list).map(tokenize_and_align, batched=False)
    silver_val_ds   = Dataset.from_list(silver_val_list).map(tokenize_and_align, batched=False)

Map:   0%|          | 0/794 [00:00<?, ? examples/s]

Map: 100%|██████████| 723/723 [00:00<00:00, 3553.50 examples/s]


## 1) Train: Silver-only

In [11]:

# ---- 1) Silver-only (evaluate on GOLD test ONLY) ----
silver_only_metrics = {}
if have_silver:
    trainer_silver = make_trainer_silver(OUTDIR_SILVER_ONLY, num_labels, id2label, label2id,
                                  train_ds=silver_train_ds, val_ds=silver_val_ds,
                                  model_name_or_path=BASE_MODEL)
    print("Training silver-only...")
    _ = trainer_silver.train()

    print("\nEvaluating silver-only on GOLD test:")
    silver_only_metrics["gold_test"] = evaluate_report(trainer_silver, gold_test_ds, id2label)
else:
    print("Skipping silver-only (no silver data).")

Some weights of BertForTokenClassification were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\hnan\AppData\Local\Temp\ipykernel_9080\1012222979.py:165: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Training silver-only...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.125600,0.057023,0.943570,0.948966,0.946260
2,0.040800,0.047694,0.954975,0.956445,0.955709


c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L1 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L2 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\P


Evaluating silver-only on GOLD test:


c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L9 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L10 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L3 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L1 seems not to 


Seqeval classification report:
              precision    recall  f1-score   support

           0     0.2786    0.3596    0.3140       406
           1     0.3385    0.3729    0.3548        59
          10     0.4444    0.2500    0.3200        32
           2     0.0167    0.0303    0.0215        33
           3     0.5980    0.9104    0.7219        67
           4     0.3673    0.6429    0.4675        28
           5     0.4268    0.8537    0.5691        82
           6     0.2632    0.3846    0.3125        13
           7     0.4032    0.6098    0.4854        82
           8     0.0000    0.0000    0.0000         0
           9     0.7857    0.3438    0.4783        32

   micro avg     0.3430    0.4700    0.3966       834
   macro avg     0.3566    0.4325    0.3677       834
weighted avg     0.3535    0.4700    0.3917       834



c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [13]:
# Prefer best checkpoint if it exists
ckpt = trainer_silver.state.best_model_checkpoint

if ckpt is None:
    # No checkpoint saved yet (e.g., save_strategy never triggered) → save final model
    ckpt = str(OUTDIR_SILVER_ONLY)
    trainer_silver.save_model(ckpt)       # saves model weights
    tokenizer.save_pretrained(ckpt)       # save tokenizer too (good practice)

print("Using checkpoint for silver→gold:", ckpt)

# Now load this for silver→gold
trainer_s2g = make_trainer(
    OUTDIR_SILVER_TO_GOLD, num_labels, id2label, label2id,
    train_ds=gold_train_ds, val_ds=gold_val_ds,
    model_name_or_path=ckpt
)

Using checkpoint for silver→gold: models\silver_only_xlmr


C:\Users\hnan\AppData\Local\Temp\ipykernel_9080\1012222979.py:129: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


## 2) Fine-tune: Silver → Gold

In [16]:

# ---- 2) Silver → Gold (fine-tune from silver-only; evaluate on GOLD test) ----
silver_to_gold_metrics = {}
if have_silver:
    ckpt = str(OUTDIR_SILVER_ONLY)  # load best from silver-only
    trainer_s2g = make_trainer(OUTDIR_SILVER_TO_GOLD, num_labels, id2label, label2id,
                               train_ds=gold_train_ds, val_ds=gold_val_ds,
                               model_name_or_path=ckpt)
    print("\nFine-tuning silver→gold...")
    _ = trainer_s2g.train()

    print("\nEvaluating silver→gold on GOLD test:")
    silver_to_gold_metrics["gold_test"] = evaluate_report(trainer_s2g, gold_test_ds, id2label)
else:
    print("Skipping silver→gold (no silver data).")

C:\Users\hnan\AppData\Local\Temp\ipykernel_9080\421255047.py:129: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



Fine-tuning silver→gold...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.213200,0.128937,0.809091,0.697128,0.748948
2,0.064600,0.122513,0.891176,0.791123,0.838174
3,0.035300,0.206042,0.908497,0.725849,0.806967
4,0.023400,0.148394,0.892754,0.804178,0.846154
5,0.011000,0.198306,0.917933,0.788512,0.848315
6,0.011500,0.184542,0.903904,0.785901,0.840782
7,0.007500,0.172833,0.858757,0.793734,0.824966
8,0.005900,0.183294,0.897059,0.796345,0.843707


c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L3 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\P


Evaluating silver→gold on GOLD test:


c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L9 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L10 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L3 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L1 seems not to 


Seqeval classification report:
              precision    recall  f1-score   support

           0     0.8734    0.8670    0.8702       406
           1     0.8750    0.9492    0.9106        59
          10     0.8333    0.9375    0.8824        32
           2     0.7568    0.8485    0.8000        33
           3     0.9077    0.8806    0.8939        67
           4     0.6875    0.7857    0.7333        28
           5     0.9726    0.8659    0.9161        82
           6     0.9000    0.6923    0.7826        13
           7     0.9605    0.8902    0.9241        82
           9     0.8611    0.9688    0.9118        32

   micro avg     0.8786    0.8765    0.8776       834
   macro avg     0.8628    0.8686    0.8625       834
weighted avg     0.8822    0.8765    0.8781       834



## 3) Train: Gold-only

In [17]:

# ---- 3) Gold-only (evaluate on GOLD test) ----
gold_only_metrics = {}
trainer_gold = make_trainer(OUTDIR_GOLD_ONLY, num_labels, id2label, label2id,
                            train_ds=gold_train_ds, val_ds=gold_val_ds,
                            model_name_or_path=BASE_MODEL)
print("\nTraining gold-only...")
_ = trainer_gold.train()

print("\nEvaluating gold-only on GOLD test:")
gold_only_metrics["gold_test"] = evaluate_report(trainer_gold, gold_test_ds, id2label)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\hnan\AppData\Local\Temp\ipykernel_9080\421255047.py:129: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



Training gold-only...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.329200,0.166343,0.794393,0.665796,0.724432
2,0.090700,0.147184,0.896226,0.744125,0.813124
3,0.049100,0.206129,0.892405,0.736292,0.806867
4,0.028900,0.184249,0.900875,0.806789,0.851240
5,0.020400,0.213537,0.889881,0.780679,0.831711
6,0.014300,0.183905,0.864407,0.798956,0.830393
7,0.009000,0.195554,0.884393,0.798956,0.839506


c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L3 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\P


Evaluating gold-only on GOLD test:


c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L9 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L10 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L3 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L1 seems not to 


Seqeval classification report:
              precision    recall  f1-score   support

           0     0.8221    0.8424    0.8321       406
           1     0.9032    0.9492    0.9256        59
          10     0.7778    0.8750    0.8235        32
           2     0.7368    0.8485    0.7887        33
           3     0.8472    0.9104    0.8777        67
           4     0.6667    0.7857    0.7213        28
           5     0.9459    0.8537    0.8974        82
           6     0.8889    0.6154    0.7273        13
           7     0.9359    0.8902    0.9125        82
           9     0.8611    0.9688    0.9118        32

   micro avg     0.8419    0.8621    0.8519       834
   macro avg     0.8386    0.8539    0.8418       834
weighted avg     0.8455    0.8621    0.8524       834



## Comparison

In [18]:

import pandas as pd

rows = []

def row_from_metrics(name, m):
    if m is None: 
        return
    # Try to get core metrics; Trainer.evaluate returns dict with keys like eval_loss, eval_precision, etc.
    prec = m.get("eval_precision") or m.get("precision")
    rec  = m.get("eval_recall") or m.get("recall")
    f1   = m.get("eval_f1") or m.get("f1")
    rows.append({"model": name, "precision": prec, "recall": rec, "f1": f1})

# Gold test comparisons
row_from_metrics("gold-only (gold test)", gold_only_metrics.get("gold_test"))
if 'gold_test' in (silver_only_metrics or {}): row_from_metrics("silver-only (gold test)", silver_only_metrics.get("gold_test"))
if 'gold_test' in (silver_to_gold_metrics or {}): row_from_metrics("silver→gold (gold test)", silver_to_gold_metrics.get("gold_test"))

df = pd.DataFrame(rows).sort_values("f1", ascending=False)
df.reset_index(drop=True, inplace=True)
df


,model,precision,recall,f1
0,silver→gold (gold test),0.878606,0.876499,0.877551
1,gold-only (gold test),0.841920,0.862110,0.851896
2,silver-only (gold test),0.342957,0.470024,0.396560


In [19]:
# --- Per-feature NER metrics (recipient, authority, legal object, legal action, legal basis) ---

import re
import pandas as pd
from typing import Dict, Any, List

def _unwrap_split_metrics(m: Dict[str, Any]) -> Dict[str, Any]:
    """If metrics dict is nested by split (e.g., {'gold_test': {...}}), unwrap to the split."""
    if not isinstance(m, dict):
        return {}
    if "gold_test" in m and isinstance(m["gold_test"], dict):
        return m["gold_test"]
    if "test" in m and isinstance(m["test"], dict):
        return m["test"]
    return m

def extract_per_label_metrics(m: Dict[str, Any]) -> Dict[str, Dict[str, float]]:
    """
    Return: {label_name: {"precision": p, "recall": r, "f1": f}}
    Tries common layouts: nested dicts or flat keys like 'eval_B-RECIPIENT_f1'.
    """
    if m is None:
        return {}
    m = _unwrap_split_metrics(m)
    out: Dict[str, Dict[str, float]] = {}

    # 1) Nested dicts under known keys
    for key in ("per_label", "labels", "by_label", "label_metrics"):
        if key in m and isinstance(m[key], dict):
            for lbl, vals in m[key].items():
                if isinstance(vals, dict):
                    p = vals.get("precision"); r = vals.get("recall"); f = vals.get("f1")
                    if p is not None or r is not None or f is not None:
                        out.setdefault(lbl, {})
                        if p is not None: out[lbl]["precision"] = p
                        if r is not None: out[lbl]["recall"] = r
                        if f is not None: out[lbl]["f1"] = f

    # 2) Flat keys: "eval_<LABEL>_precision", "<LABEL>_recall", etc.
    for k, v in m.items():
        if not isinstance(k, str):
            continue
        mobj = re.match(r"^(?:eval_)?(.+?)_(precision|recall|f1)$", k)
        if mobj:
            lbl = mobj.group(1)
            metric = mobj.group(2)
            out.setdefault(lbl, {})[metric] = v

    return out

def _normalize_id2label(id2label_obj) -> Dict[int, str]:
    """Accepts HF-style dict or list and returns {int_id: label_str}."""
    if id2label_obj is None:
        return {}
    if isinstance(id2label_obj, dict):
        # Convert keys to int if possible
        norm = {}
        for k, v in id2label_obj.items():
            try:
                norm[int(k)] = str(v)
            except Exception:
                # ignore non-int keys
                pass
        return norm
    if isinstance(id2label_obj, (list, tuple)):
        return {i: str(lbl) for i, lbl in enumerate(id2label_obj)}
    return {}

def build_feature_groups(id2label: Dict[int, str] = None) -> Dict[str, List[str]]:
    """
    Features → their two label NER IDs (1–10). If id2label is provided,
    map to actual tag names; else fall back to "1","2",... strings.
    """
    pairs = [
        ("recipient", (1, 2)),
        ("authority", (3, 4)),
        ("legal object", (5, 6)),
        ("legal action", (7, 8)),
        ("legal basis", (9, 10)),
    ]
    id2label = _normalize_id2label(id2label)
    fmap: Dict[str, List[str]] = {}
    for feat, (i, j) in pairs:
        if i in id2label and j in id2label:
            fmap[feat] = [id2label[i], id2label[j]]
        else:
            fmap[feat] = [str(i), str(j)]
    return fmap

def feature_rows_from_metrics(model_name: str, m: Dict[str, Any], feature_map: Dict[str, List[str]]) -> List[Dict[str, Any]]:
    """
    Macro-average precision/recall/F1 across the two labels in each feature.
    """
    per_label = extract_per_label_metrics(m)
    rows: List[Dict[str, Any]] = []
    if not per_label:
        return rows

    for feat, lbls in feature_map.items():
        vals = [per_label.get(lbl) for lbl in lbls if per_label.get(lbl)]
        if not vals:
            continue
        p = sum(v.get("precision", 0.0) for v in vals) / len(vals)
        r = sum(v.get("recall", 0.0) for v in vals) / len(vals)
        f = sum(v.get("f1", 0.0) for v in vals) / len(vals)
        rows.append({"model": model_name, "feature": feat, "precision": p, "recall": r, "f1": f})
    return rows

# ---- Build table for your three setups (uses the same variables you already have) ----
rows_feat: List[Dict[str, Any]] = []

# Use global id2label if defined (HF-style), otherwise fallback to numeric IDs
feature_map = build_feature_groups(globals().get("id2label"))

if "gold_only_metrics" in globals() and isinstance(gold_only_metrics, dict):
    rows_feat += feature_rows_from_metrics("gold-only (gold test)", gold_only_metrics, feature_map)

if "silver_only_metrics" in globals() and isinstance(silver_only_metrics, dict):
    # Use the gold test slice if present for fair comparison
    rows_feat += feature_rows_from_metrics("silver-only (gold test)", silver_only_metrics, feature_map)

if "silver_to_gold_metrics" in globals() and isinstance(silver_to_gold_metrics, dict):
    rows_feat += feature_rows_from_metrics("silver→gold (gold test)", silver_to_gold_metrics, feature_map)

df_features = pd.DataFrame(rows_feat)
if not df_features.empty:
    df_features = df_features.sort_values(["feature", "f1"], ascending=[True, False]).reset_index(index=False)
else:
    df_features = pd.DataFrame([{
        "model": "(no per-label metrics found)", "feature": "(n/a)",
        "precision": None, "recall": None, "f1": None
    }])

# View
df_features


,model,feature,precision,recall,f1
0,(no per-label metrics found),(n/a),None,None,None


In [20]:
def evaluate_report2(trainer: "Trainer", test_ds, id2label) -> dict:
    """
    Prints a seqeval classification report at the FEATURE/entity level
    (BIO-aware, groups B/I under the same feature). Returns the original
    trainer.evaluate(test_ds) dict unchanged.
    """
    # 1) Evaluate to keep identical training stats
    metrics = trainer.evaluate(test_ds)

    # 2) Predict logits -> argmax ids
    pred_out = trainer.predict(test_ds)
    preds = np.argmax(pred_out.predictions, axis=-1)
    labels = test_ds["labels"]

    # 3) Normalize id2label to a dict[int,str]
    if isinstance(id2label, dict):
        id2lab = {int(k): str(v) for k, v in id2label.items()}
    else:
        # assume list/tuple
        id2lab = {i: str(v) for i, v in enumerate(id2label)}

    # 4) Map ids to BIO tags (prefer existing BIO in id2label; otherwise fallback to 1–10 mapping)
    fallback_map = {
        0: "O",
        1: "B-RECIPIENT",  2: "I-RECIPIENT",
        3: "B-AUTHORITY",  4: "I-AUTHORITY",
        5: "B-LEGAL_OBJECT", 6: "I-LEGAL_OBJECT",
        7: "B-LEGAL_ACTION", 8: "I-LEGAL_ACTION",
        9: "B-LEGAL_BASIS", 10: "I-LEGAL_BASIS",
    }

    def to_bio(label_id: int) -> str:
        tag = id2lab.get(int(label_id), None)
        if isinstance(tag, str):
            # already looks like BIO or 'O'?
            if tag == "O" or tag.startswith("B-") or tag.startswith("I-"):
                return tag
        # fallback numeric mapping
        return fallback_map.get(int(label_id), "O")

    # 5) Build seqeval-compatible BIO tag sequences (mask out -100)
    y_true, y_pred = [], []
    for pred_ids, lab_ids in zip(preds, labels):
        tt, pp = [], []
        for p_id, l_id in zip(pred_ids, lab_ids):
            if l_id == -100:
                continue
            tt.append(to_bio(l_id))
            pp.append(to_bio(p_id))
        y_true.append(tt)
        y_pred.append(pp)

    # 6) Print FEATURE/entity-level classification report (seqeval is entity-aware)
    print("\nSeqeval classification report (entity-level, grouped by feature):")
    print(classification_report(y_true, y_pred, digits=4))

    # Optional: show overall micro feature-level P/R/F1
    print("Feature-level micro P/R/F1:",
          f"{precision_score(y_true, y_pred):.4f}",
          f"{recall_score(y_true, y_pred):.4f}",
          f"{f1_score(y_true, y_pred):.4f}")

    return metrics

In [25]:
print('gold:')
gold_only_metrics["gold_test"] = evaluate_report2(trainer_gold, gold_test_ds, id2label)

print('silver:')
silver_only_metrics["gold_test"] = evaluate_report2(trainer_silver, gold_test_ds, id2label)

print('silver to gold:')
silver_to_gold_metrics["gold_test"] = evaluate_report2(trainer_s2g, gold_test_ds, id2label)

gold:



Seqeval classification report (entity-level, grouped by feature):
              precision    recall  f1-score   support

   AUTHORITY     0.7733    0.8657    0.8169        67
LEGAL_ACTION     0.9359    0.8902    0.9125        82
 LEGAL_BASIS     0.7568    0.8750    0.8116        32
LEGAL_OBJECT     0.9333    0.8537    0.8917        82
   RECIPIENT     0.7910    0.8983    0.8413        59

   micro avg     0.8494    0.8758    0.8624       322
   macro avg     0.8381    0.8766    0.8548       322
weighted avg     0.8571    0.8758    0.8642       322

Feature-level micro P/R/F1: 0.8494 0.8758 0.8624
silver:

Seqeval classification report (entity-level, grouped by feature):
              precision    recall  f1-score   support

   AUTHORITY     0.5670    0.8209    0.6707        67
LEGAL_ACTION     0.4098    0.6098    0.4902        82
 LEGAL_BASIS     0.5333    0.2500    0.3404        32
LEGAL_OBJECT     0.4268    0.8537    0.5691        82
   RECIPIENT     0.2211    0.3559    0.2727      


Seqeval classification report (entity-level, grouped by feature):
              precision    recall  f1-score   support

   AUTHORITY     0.8235    0.8358    0.8296        67
LEGAL_ACTION     0.9605    0.8902    0.9241        82
 LEGAL_BASIS     0.8333    0.9375    0.8824        32
LEGAL_OBJECT     0.9726    0.8659    0.9161        82
   RECIPIENT     0.8308    0.9153    0.8710        59

   micro avg     0.8931    0.8820    0.8875       322
   macro avg     0.8842    0.8889    0.8846       322
weighted avg     0.8987    0.8820    0.8885       322

Feature-level micro P/R/F1: 0.8931 0.8820 0.8875


In [26]:
from typing import List, Tuple, Dict, Set
from collections import Counter, defaultdict

Span = Tuple[int, int, str]  # (start_idx, end_idx_exclusive, label)

def tags_to_spans(tags: List[str], scheme: str = "BIO") -> List[Span]:
    """
    Convert a single sequence of BIO/BILOU tags into (start, end, label) spans.
    end is exclusive. Ignores 'O'.
    Heals some broken BIO like 'I-X' without a preceding 'B-X' by starting a new span.
    """
    spans: List[Span] = []
    start, label = None, None

    def close_span(i):
        nonlocal start, label
        if start is not None and label is not None:
            spans.append((start, i, label))
        start, label = None, None

    for i, t in enumerate(tags):
        if t == "O" or t == "_" or t is None:
            close_span(i)
            continue

        # split like "B-PER", "I-ORG"
        if "-" in t:
            prefix, lab = t.split("-", 1)
        else:
            # If tag has no prefix (rare), treat as B-<t>
            prefix, lab = "B", t

        if scheme.upper().startswith("BILOU"):
            # Minimal BILOU support: treat U- as a one-token span; L- as close; I- as inside
            if prefix == "U":   # unit-length
                close_span(i)            # close any open
                spans.append((i, i+1, lab))
            elif prefix == "B":
                close_span(i)
                start, label = i, lab
            elif prefix == "I":
                # continue
                if label != lab or start is None:
                    # heal invalid I- by starting anew
                    close_span(i)
                    start, label = i, lab
            elif prefix == "L":
                if label == lab and start is not None:
                    spans.append((start, i+1, lab))
                else:
                    # heal invalid L- by treating as a single-token span
                    close_span(i)
                    spans.append((i, i+1, lab))
                start, label = None, None
            else:  # 'O' already handled above
                close_span(i)
        else:
            # BIO (default)
            if prefix == "B":
                close_span(i)
                start, label = i, lab
            elif prefix == "I":
                if label != lab or start is None:
                    # heal invalid I- by starting a new span
                    close_span(i)
                    start, label = i, lab
            else:  # unexpected prefix; close any open
                close_span(i)

    close_span(len(tags))
    return spans


In [35]:
from typing import List, Tuple, Dict, Set
from collections import defaultdict

Span = Tuple[int, int, str]  # (start_idx, end_idx_exclusive, label)

def tags_to_spans_strict(tags: List[str], scheme: str = "BIO", allowed_labels: Set[str] = None) -> List[Span]:
    """
    Strict conversion of BIO/BILOU tags to spans:
    - Ignores 'O' and any label not in allowed_labels (if provided).
    - NO healing: invalid sequences are discarded.
    - Exact boundaries only.
    """
    spans: List[Span] = []
    start, label = None, None

    def close_span(i):
        nonlocal start, label
        if start is not None and label is not None:
            spans.append((start, i, label))
        start, label = None, None

    def is_allowed(lab: str) -> bool:
        if allowed_labels is None:
            return lab != "O"
        return lab in allowed_labels

    sch = scheme.upper()
    for i, t in enumerate(tags):
        if not t or t == "O" or t == "_" or t is None:
            # End any open span
            if start is not None:
                close_span(i)
            continue

        # Expect PREFIX-LABEL
        if "-" not in t:
            # Unknown format -> end any open span and skip
            if start is not None:
                close_span(i)
            continue

        prefix, lab = t.split("-", 1)
        if not is_allowed(lab):
            # Encounter of non-allowed label ends any open span, skip token
            if start is not None:
                close_span(i)
            continue

        if sch.startswith("BILOU"):
            if prefix == "B":
                # start new
                if start is not None:  # close previous
                    close_span(i)
                start, label = i, lab
            elif prefix == "I":
                # continue only if same label and we are inside a span
                if start is None or label != lab:
                    # invalid I -> drop; do NOT heal; end any open
                    if start is not None:
                        close_span(i)
                    start, label = None, None
            elif prefix == "L":
                # close only if currently in matching span
                if start is not None and label == lab:
                    close_span(i + 1)
                # else invalid -> drop
                start, label = None, None
            elif prefix == "U":
                # unit-length span
                if start is not None:
                    close_span(i)
                spans.append((i, i + 1, lab))
                start, label = None, None
            else:  # unknown prefix -> close any open and skip
                if start is not None:
                    close_span(i)
                start, label = None, None

        else:  # BIO strict
            if prefix == "B":
                if start is not None:
                    close_span(i)
                start, label = i, lab
            elif prefix == "I":
                # continue only if we are in a span of the same label
                if start is None or label != lab:
                    # invalid I -> drop; do NOT heal
                    if start is not None:
                        close_span(i)
                    start, label = None, None
            else:
                # any other prefix invalid in BIO -> close any open and skip
                if start is not None:
                    close_span(i)
                start, label = None, None

    # close tail
    if start is not None:
        close_span(len(tags))
    return spans


def prf_exact(true_spans: Set[Span], pred_spans: Set[Span]) -> Dict[str, float]:
    tp = len(true_spans & pred_spans)
    fp = len(pred_spans - true_spans)
    fn = len(true_spans - pred_spans)
    P = tp / (tp + fp) if (tp + fp) else 0.0
    R = tp / (tp + fn) if (tp + fn) else 0.0
    F = 2 * P * R / (P + R) if (P + R) else 0.0
    return dict(tp=tp, fp=fp, fn=fn, precision=P, recall=R, f1=F)


def evaluate_spans_positive_only(
    y_true_tags: List[List[str]],
    y_pred_tags: List[List[str]],
    scheme: str = "BIO",
    allowed_labels: Set[str] = None,   # e.g., set(id2label.values()) - {"O"}
    per_type: bool = True
) -> Dict[str, float]:
    """
    Strict, positive-only span evaluation:
    - Only entity labels in allowed_labels (or != 'O' if None)
    - Exact match on (start, end, label)
    - No BIO/BILOU healing
    """
    assert len(y_true_tags) == len(y_pred_tags)
    true_all: Set[Span] = set()
    pred_all: Set[Span] = set()

    type_true: Dict[str, Set[Span]] = defaultdict(set)
    type_pred: Dict[str, Set[Span]] = defaultdict(set)

    offset = 0
    for gold_tags, pred_tags in zip(y_true_tags, y_pred_tags):
        g_spans = [(s + offset, e + offset, lab) for (s, e, lab) in tags_to_spans_strict(gold_tags, scheme, allowed_labels)]
        p_spans = [(s + offset, e + offset, lab) for (s, e, lab) in tags_to_spans_strict(pred_tags, scheme, allowed_labels)]

        true_all.update(g_spans)
        pred_all.update(p_spans)
        for s, e, lab in g_spans:
            type_true[lab].add((s, e, lab))
        for s, e, lab in p_spans:
            type_pred[lab].add((s, e, lab))

        offset += len(gold_tags)

    agg = prf_exact(true_all, pred_all)
    out = {
        "span_precision_micro": agg["precision"],
        "span_recall_micro": agg["recall"],
        "span_f1_micro": agg["f1"],
        "span_tp": agg["tp"],
        "span_fp": agg["fp"],
        "span_fn": agg["fn"],
        "span_support": len(true_all),   # total gold entities (positive-only)
        "span_predicted": len(pred_all),
    }

    if per_type:
        for lab in sorted(set(list(type_true.keys()) + list(type_pred.keys()))):
            d = prf_exact(type_true[lab], type_pred[lab])
            out[f"span_precision_{lab}"] = d["precision"]
            out[f"span_recall_{lab}"] = d["recall"]
            out[f"span_f1_{lab}"] = d["f1"]
            out[f"span_support_{lab}"] = len(type_true[lab])
            out[f"span_predicted_{lab}"] = len(type_pred[lab])

    return out

In [ ]:
rows = []
for name, trainer in [
    ("gold-only", trainer_gold),
    ("silver-only", trainer_silver),
    ("silver→gold", trainer_s2g),
]:
    predictions = trainer.predict(gold_test_ds)
    preds = np.argmax(predictions.predictions, axis=-1)
    labels = predictions.label_ids

    true_labels, true_predictions = [], []
    for pred, lab in zip(preds, labels):
        cur_preds, cur_labels = [], []
        for p_id, l_id in zip(pred, lab):
            if l_id != -100:
                cur_preds.append(id2label[p_id])
                cur_labels.append(id2label[l_id])
        true_predictions.append(cur_preds)
        true_labels.append(cur_labels)

    res = evaluate_spans(true_labels, true_predictions, scheme="BIO", per_type=True)
    rows.append({
        "model": name,
        "span_P": res["span_precision_micro"],
        "span_R": res["span_recall_micro"],
        "span_F1": res["span_f1_micro"],
    })

import pandas as pd
pd.DataFrame(rows)

c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L9 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L10 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L3 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L1 seems not to 

,model,span_P,span_R,span_F1
0,gold-only,0.973367,0.973367,0.973367
1,silver-only,0.848048,0.848048,0.848048
2,silver→gold,0.976286,0.976286,0.976286


In [50]:
def bio_to_spans(tags):
    """Convert BIO tag sequence into spans: list of (type, start_idx, end_idx_inclusive)."""
    spans = []
    cur_type, start = None, None
    for i, t in enumerate(tags):
        if t == "O" or t is None:
            if cur_type is not None:
                spans.append((cur_type, start, i-1))
                cur_type, start = None, None
            continue
        prefix, etype = (t.split("-", 1)+[None])[:2]
        if prefix == "B" or (prefix == "I" and (cur_type != etype)):
            if cur_type is not None:
                spans.append((cur_type, start, i-1))
            cur_type, start = etype, i
        elif prefix == "I" and cur_type == etype:
            pass
        else:
            # Any irregularity closes current span
            if cur_type is not None:
                spans.append((cur_type, start, i-1))
            cur_type, start = None, None
    if cur_type is not None:
        spans.append((cur_type, start, len(tags)-1))
    return spans

def span_level_diffs(trainer, dataset, id2label, n_sentences=5):
    predictions = trainer.predict(dataset)
    preds = np.argmax(predictions.predictions, axis=-1)
    labels = predictions.label_ids

    rows = []
    for i, (pred, lab) in enumerate(zip(preds, labels)):
        gold_tags = [id2label[l] for l in lab if l != -100]
        pred_tags = [id2label[p] for p, l in zip(pred, lab) if l != -100]

        gold_spans = set(bio_to_spans(gold_tags))
        pred_spans = set(bio_to_spans(pred_tags))

        tp = gold_spans & pred_spans
        fp = pred_spans - gold_spans
        fn = gold_spans - pred_spans

        def fmt(sp): return [{"type": t, "start": s, "end": e} for (t,s,e) in sorted(sp)]

        rows.append({
            "sentence_id": i,
            "TP_spans": fmt(tp),
            "FP_spans": fmt(fp),
            "FN_spans": fmt(fn),
        })
        if i+1 >= n_sentences:
            break

    return pd.DataFrame(rows)

# Example: inspect span-level errors for one model
display(span_level_diffs(trainer_s2g, gold_test_ds, id2label, n_sentences=50))


c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L9 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L10 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L3 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
c:\Users\hnan\AppData\Local\Programs\Python\Python310\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L1 seems not to 

,sentence_id,TP_spans,FP_spans,FN_spans
0,0,[],[],[]
1,1,[],[],[]
2,2,[],[],[]
3,3,[],[],[]
4,4,[],[],[]
5,5,[],[],[]
6,6,[],[],[]
7,7,[],[],[]
8,8,[],[],[]
9,9,[],[],[]


In [41]:
true_labels = []
true_predictions = []

for pred, lab in zip(preds, labels):
    cur_preds, cur_labels = [], []
    for p_id, l_id in zip(pred, lab):
        if l_id != -100:   # ignore subtoken positions
            cur_preds.append(id2label[p_id])
            cur_labels.append(id2label[l_id])
    true_predictions.append(cur_preds)
    true_labels.append(cur_labels)

In [42]:
# Build whitelist: all entity labels except 'O'
allowed_labels = set(id2label.values()) - {"O"}

span_results = evaluate_spans_positive_only(
    true_labels,           # gold tag sequences (list of lists of BIO strings)
    true_predictions,      # predicted tag sequences
    scheme="BIO",          # or "BILOU" if that's your scheme
    allowed_labels=allowed_labels,
    per_type=True          # <- this makes it compute per-entity type
)


In [43]:
import pandas as pd

# Collect per-type rows
rows = []
for lab in allowed_labels:
    rows.append({
        "label": lab,
        "precision": span_results.get(f"span_precision_{lab}", 0.0),
        "recall": span_results.get(f"span_recall_{lab}", 0.0),
        "f1": span_results.get(f"span_f1_{lab}", 0.0),
        "support": span_results.get(f"span_support_{lab}", 0),
        "predicted": span_results.get(f"span_predicted_{lab}", 0),
    })

df = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)
df


,label,precision,recall,f1,support,predicted
0,L10,0.0,0.0,0.0,0,0
1,L7,0.0,0.0,0.0,0,0
2,L1,0.0,0.0,0.0,0,0
3,L9,0.0,0.0,0.0,0,0
4,L8,0.0,0.0,0.0,0,0
5,L3,0.0,0.0,0.0,0,0
6,L2,0.0,0.0,0.0,0,0
7,L0,0.0,0.0,0.0,0,0
8,L6,0.0,0.0,0.0,0,0
9,L4,0.0,0.0,0.0,0,0


In [ ]:
# === Notebook 1: EXPORT FIRST-5 PREDICTIONS TO /mnt/data/nb1_preds.jsonl ===
import json, numpy as np
from typing import List, Tuple, Dict, Set
from collections import defaultdict

# ---------- CONFIG: name your 3 models here ----------
MODELS_NB1 = [
    ("nb1_gold_only",   "trainer_gold"),
    ("nb1_silver_only", "trainer_silver"),
    ("nb1_silver_to_gold", "trainer_s2g"),
]
TEST_DATASET = gold_test_ds

# ---------- Helpers: BIO/BILOU -> spans (strict, positive-only) ----------
Span = Tuple[int, int, str]  # (start_idx, end_idx_exclusive, label)

def tags_to_spans_strict(tags: List[str], scheme: str = "BIO", allowed_labels: Set[str] = None) -> List[Span]:
    spans: List[Span] = []
    start, label = None, None

    def close_span(i):
        nonlocal start, label
        if start is not None and label is not None:
            spans.append((start, i, label))
        start, label = None, None

    def is_allowed(lab: str) -> bool:
        if allowed_labels is None:
            return lab != "O"
        return lab in allowed_labels

    sch = scheme.upper()
    for i, t in enumerate(tags):
        if not t or t in ("O", "_", None):
            if start is not None:
                close_span(i)
            continue
        if "-" not in t:
            if start is not None:
                close_span(i)
            continue
        prefix, lab = t.split("-", 1)
        if not is_allowed(lab):
            if start is not None:
                close_span(i)
            continue

        if sch.startswith("BILOU"):
            if prefix == "B":
                if start is not None:
                    close_span(i)
                start, label = i, lab
            elif prefix == "I":
                if start is None or label != lab:
                    if start is not None:
                        close_span(i)
                    start, label = None, None
            elif prefix == "L":
                if start is not None and label == lab:
                    close_span(i+1)
                start, label = None, None
            elif prefix == "U":
                if start is not None:
                    close_span(i)
                spans.append((i, i+1, lab))
                start, label = None, None
            else:
                if start is not None:
                    close_span(i)
                start, label = None, None
        else:
            if prefix == "B":
                if start is not None:
                    close_span(i)
                start, label = i, lab
            elif prefix == "I":
                if start is None or label != lab:
                    if start is not None:
                        close_span(i)
                    start, label = None, None
            else:
                if start is not None:
                    close_span(i)
                start, label = None, None

    if start is not None:
        close_span(len(tags))
    return spans

def render_annotated(tokens: List[str], spans: List[Span]) -> str:
    """
    Render like: The [College|BESLUITVORMEND_ORGAAN] besluit ...
    """
    spans = sorted(spans, key=lambda x: (x[0], -x[1]))
    out = []
    i = 0
    for (s, e, lab) in spans:
        if s > i:
            out.extend(tokens[i:s])
        out.append("[" + " ".join(tokens[s:e]) + "|" + lab + "]")
        i = e
    if i < len(tokens):
        out.extend(tokens[i:])
    # space-join, then clean spaces before punctuation if needed
    text = " ".join(out)
    text = text.replace(" ,", ",").replace(" .", ".").replace(" )", ")").replace("( ", "(")
    return text

# Attempt to fetch word tokens & gold tags from the dataset item
def get_tokens_and_gold_tags_from_item(item) -> Tuple[List[str], List[str]]:
    # Common HF formats:
    if "tokens" in item and "ner_tags" in item:
        tokens = list(item["tokens"])
        # map tag ids -> strings if needed
        if isinstance(item["ner_tags"][0], int):
            gold_tags = [id2label[t] for t in item["ner_tags"]]
        else:
            gold_tags = list(item["ner_tags"])
        return tokens, gold_tags

    # Fallback: some datasets store "words" instead of "tokens"
    if "words" in item and "ner_tags" in item:
        tokens = list(item["words"])
        if isinstance(item["ner_tags"][0], int):
            gold_tags = [id2label[t] for t in item["ner_tags"]]
        else:
            gold_tags = list(item["ner_tags"])
        return tokens, gold_tags

    raise ValueError("Could not find 'tokens'/'words' and 'ner_tags' in dataset item; adapt this helper.")

# Predict tags with a Trainer, then collapse to word-level strings
def predict_word_level_tags(trainer, dataset_slice):
    predictions = trainer.predict(dataset_slice)
    preds = np.argmax(predictions.predictions, axis=-1)
    labels = predictions.label_ids

    # collapse subtoken positions (ignore -100)
    seq_preds, seq_labels = [], []
    for pred, lab in zip(preds, labels):
        cur_preds, cur_labels = [], []
        for p_id, l_id in zip(pred, lab):
            if l_id != -100:
                cur_preds.append(id2label[p_id])
                cur_labels.append(id2label[l_id])
        seq_preds.append(cur_preds)
        seq_labels.append(cur_labels)
    return seq_preds, seq_labels

# Build whitelist: everything except 'O'
allowed_labels = set(id2label.values()) - {"O"}

# Take the first 5 examples (adjust if test set shorter)
N = min(5, len(TEST_DATASET))
subset = TEST_DATASET.select(range(N))

# Run all 3 models and export
all_output = []  # one JSON record per example index
# Get gold tokens/tags from the underlying dataset (not from the trainer)
gold_word_tokens = []
gold_word_tags   = []
for i in range(N):
    item = TEST_DATASET[i]
    toks, gtags = get_tokens_and_gold_tags_from_item(item)
    gold_word_tokens.append(toks)
    gold_word_tags.append(gtags)

# Precompute gold spans and a rendered gold sentence
gold_spans = [tags_to_spans_strict(gtags, "BIO", allowed_labels) for gtags in gold_word_tags]
gold_rendered = [render_annotated(toks, sp) for toks, sp in zip(gold_word_tokens, gold_spans)]

# Collect predictions for each model
per_model_rendered = defaultdict(list)

for alias, trainer_var in MODELS_NB1:
    trainer = globals().get(trainer_var, None)
    if trainer is None:
        # still store an empty list with a note
        per_model_rendered[alias] = ["<trainer not found: {}>".format(trainer_var)] * N
        continue
    seq_preds, _ = predict_word_level_tags(trainer, subset)
    pred_spans   = [tags_to_spans_strict(p, "BIO", allowed_labels) for p in seq_preds]
    rendered     = [render_annotated(toks, sp) for toks, sp in zip(gold_word_tokens, pred_spans)]
    per_model_rendered[alias] = rendered

# Build JSONL
for i in range(N):
    rec = {
        "index": i,
        "tokens": gold_word_tokens[i],
        "gold_rendered": gold_rendered[i],
        "nb1_gold_only":     per_model_rendered.get("nb1_gold_only", [])[i] if per_model_rendered.get("nb1_gold_only") else None,
        "nb1_silver_only":   per_model_rendered.get("nb1_silver_only", [])[i] if per_model_rendered.get("nb1_silver_only") else None,
        "nb1_silver_to_gold":per_model_rendered.get("nb1_silver_to_gold", [])[i] if per_model_rendered.get("nb1_silver_to_gold") else None,
    }
    all_output.append(rec)

out_path = "nb1_preds.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for rec in all_output:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Wrote {len(all_output)} examples to {out_path}")


Wrote 5 examples to nb1_preds.jsonl
